# Creating Widgets
- Widgets make the notebook accept input values
- Later ADF can call this same notebook with different table/file names
- We avoid creating 9 separate notebooks

In [0]:

dbutils.widgets.text("source_system", "")
dbutils.widgets.text("source_object", "")
dbutils.widgets.text("landing_path", "")
dbutils.widgets.text("bronze_path", "")
dbutils.widgets.text("load_id", "")


In [0]:

source_system = dbutils.widgets.get("source_system")
source_object = dbutils.widgets.get("source_object")
landing_path = dbutils.widgets.get("landing_path")
bronze_path = dbutils.widgets.get("bronze_path")
load_id = dbutils.widgets.get("load_id")


In [0]:
df = spark.read.format("parquet").load(landing_path)


In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_df = (
    df
    .withColumn("LoadID", lit(load_id))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit(source_system))
    .withColumn("source_object", lit(source_object))
)

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)


In [0]:

landing_df = spark.read.parquet(landing_path)
bronze_check = spark.read.format("delta").load(bronze_path)

landing_count = landing_df.count()
bronze_count = bronze_check.count()

print("Source system:", source_system)
print("Source object:", source_object)
print("Landing rows:", landing_count)
print("Bronze rows:", bronze_count)

bronze_check.select(
    "LoadID",
    "ingestion_timestamp",
    "source_system",
    "source_object"
).show(5, truncate=False)

assert landing_count == bronze_count, "Landing/Bronze row count mismatch"

assert source_system.strip(), "Missing source_system"
assert source_object.strip(), "Missing source_object"

print("Bronze validation PASSED")


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7977335110892254>, line 1
----> 1 landing_df = spark.read.parquet(landing_path)
      2 bronze_check = spark.read.format("delta").load(bronze_path)
      4 landing_count = landing_df.count()

NameError: name 'landing_path' is not defined